# Programação dinâmica em grafos: os algoritmos de Warshall e de Floyd — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Este tutorial acompanha a aula. O código dos algoritmos é escrito em C e compilado com `gcc`
a partir do próprio notebook: cada célula `%%writefile` grava um programa em `src/`, e a célula
seguinte compila e executa. As medições, os gráficos e alguns exercícios ficam em Python.

## Objetivos

Ao final deste tutorial você será capaz de:

- calcular o **fecho transitivo** de um digrafo com uma busca por vértice e usá-lo para achar as componentes fortes;
- implementar o algoritmo de **Warshall** em uma única matriz e acompanhar as matrizes $R^{(k)}$ rodada a rodada;
- implementar o algoritmo de **Floyd** tratando o $\infty$ com cuidado, com pesos negativos e detecção de **ciclos negativos**;
- **reconstruir** caminhos mínimos com a matriz *prox*;
- explicar, com um contraexemplo, por que o laço de $k$ tem de ser o **externo**.

In [ ]:
# Preparação do ambiente: pasta de trabalho, compilador e funções auxiliares.
import os, subprocess, math

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

_compilados = {}   # fonte -> data de modificação da última compilação bem-sucedida

def compilar(fonte):
    """Compila src/<fonte> com gcc -O2 (só se o arquivo mudou); devolve o executável ou None."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    mtime = os.path.getmtime(os.path.join('src', fonte))
    if _compilados.get(fonte) == mtime and os.path.exists(exe):
        return exe
    comp = subprocess.run(['gcc', '-std=c99', '-Wall', '-O2', '-o', exe,
                           os.path.join('src', fonte), '-lm'],
                          capture_output=True, text=True)
    if comp.returncode != 0:
        print('ERRO DE COMPILAÇÃO:\n' + comp.stderr)
        return None
    if comp.stderr:
        print(comp.stderr)
    _compilados[fonte] = mtime
    return exe

def rodar(fonte, args=(), entrada=None, mostrar=True):
    """Compila src/<fonte>, executa com os argumentos e a entrada dados e devolve a saída."""
    exe = compilar(fonte)
    if exe is None:
        return ''
    r = subprocess.run([exe, *map(str, args)], input=entrada, capture_output=True, text=True,
                       errors='replace')
    if mostrar:
        print(r.stdout, end='')
    if r.returncode != 0:
        print(f'(programa terminou com código {r.returncode})', r.stderr)
    return r.stdout

def campo(saida, nome):
    """Extrai o valor de 'nome=valor' da saída (int se for inteiro, float se for número, senão str)."""
    for tok in saida.split():
        if tok.startswith(nome + '='):
            valor = tok.split('=', 1)[1]
            if valor.lstrip('-').isdigit():
                return int(valor)
            try:
                return float(valor)
            except ValueError:
                return valor
    raise ValueError(f'{nome} não encontrado em: {saida!r}')

def linha(saida, rotulo):
    """Devolve os números da linha que começa com 'rotulo:' ('INF' vira math.inf)."""
    for lin in saida.splitlines():
        if lin.startswith(rotulo + ':'):
            return [math.inf if x == 'INF' else int(x) for x in lin.split(':', 1)[1].split()]
    raise ValueError(f'linha {rotulo} não encontrada em: {saida!r}')

def matriz(saida, prefixo, n):
    """Lê as n linhas '<prefixo>0:', '<prefixo>1:', ... como uma lista de listas."""
    return [linha(saida, f'{prefixo}{i}') for i in range(n)]

def entrada_grafo(n, arcos):
    """Monta a entrada dos programas em C: "n m" e depois um arco por linha ("u v" ou "u v w")."""
    return f'{n} {len(arcos)}\n' + ''.join(' '.join(map(str, a)) + '\n' for a in arcos)

INF = math.inf

# Digrafo das aulas de grafos (Feofiloff): 0-1 0-5 1-0 1-5 2-4 3-1 5-3.
N_F, ARCOS_F = 6, [(0, 1), (0, 5), (1, 0), (1, 5), (2, 4), (3, 1), (5, 3)]
# Exemplo de Floyd do Levitin (2012, §8.4): a, b, c, d = 0, 1, 2, 3.
N_L, ARCOS_L = 4, [(0, 2, 3), (1, 0, 2), (2, 1, 7), (2, 3, 1), (3, 0, 6)]
# Exemplo com pesos negativos do Cormen (2009, Figura 25.4), vértices 1..5 renumerados para 0..4.
N_C, ARCOS_C = 5, [(0, 1, 3), (0, 2, 8), (0, 4, -4), (1, 3, 1), (1, 4, 7),
                   (2, 1, 4), (3, 0, 2), (3, 2, -5), (4, 3, 6)]

## 1. Fecho transitivo por buscas

O **fecho transitivo** de um digrafo é a matriz $R$ com $R[i][j] = 1$ quando existe caminho
(com pelo menos um arco) de $i$ até $j$. A forma mais direta de calculá-lo é a que já conhecemos
das aulas de grafos: uma busca a partir de cada vértice, em $O(n(n+m))$ no total.

Atenção à diagonal: $R[i][i] = 1$ só se $i$ está num **ciclo**, porque o caminho precisa ter
pelo menos um arco. Por isso a busca abaixo começa pelos *vizinhos* de $s$, e não por $s$.

**Fonte:** Cormen et al. (2009), §25.2 ("Transitive closure of a directed graph").

In [ ]:
# Exemplo: fecho transitivo com uma BFS a partir de cada vértice.
from collections import deque

def fecho_bfs(n, arcos):
    adj = [[] for _ in range(n)]
    for u, v in arcos:
        adj[u].append(v)
    R = [[0] * n for _ in range(n)]
    for s in range(n):
        fila = deque(adj[s])          # começa pelos vizinhos: R[s][s] = 1 só se houver ciclo
        while fila:
            v = fila.popleft()
            if not R[s][v]:
                R[s][v] = 1
                fila.extend(adj[v])
    return R

R_F = fecho_bfs(N_F, ARCOS_F)
for i, lin in enumerate(R_F):
    print(i, ':', *lin)

### Exercício 1: componentes fortes a partir do fecho

Dois vértices $i \ne j$ estão na mesma **componente fortemente conexa** quando cada um alcança o
outro: $R[i][j] = R[j][i] = 1$. Complete `componentes_fortes(R)`, que recebe o fecho e devolve a
lista das componentes, cada uma como um `set`, em qualquer ordem. Todo vértice pertence a
exatamente uma componente (às vezes sozinho). No grafo das aulas de grafos, a resposta é
$\{0,1,3,5\}$, $\{2\}$ e $\{4\}$.

In [ ]:
# Exercício 1: componentes fortes a partir do fecho transitivo
# TODO: implemente componentes_fortes(R), que devolve uma lista de conjuntos de vértices.
# Dica: percorra os vértices; cada vértice ainda sem componente abre uma nova, que recebe
# todos os j com R[i][j] and R[j][i].

def componentes_fortes(R):
    # TODO: implemente aqui
    pass


# Teste automático
import random
assert componentes_fortes(R_F) is not None, 'Implemente componentes_fortes()'
assert sorted(map(sorted, componentes_fortes(R_F))) == [[0, 1, 3, 5], [2], [4]]
random.seed(1)
for _ in range(300):
    n = random.randint(1, 9)
    arcos = [(u, v) for u in range(n) for v in range(n) if u != v and random.random() < 0.2]
    R = fecho_bfs(n, arcos)
    comps = componentes_fortes(R)
    assert sorted(v for c in comps for v in c) == list(range(n)), 'Cada vértice em exatamente uma componente'
    qual = {v: k for k, c in enumerate(comps) for v in c}
    for i in range(n):
        for j in range(n):
            if i != j:
                assert (qual[i] == qual[j]) == bool(R[i][j] and R[j][i]), f'Falhou com arcos={arcos}'
print('Exercício 1 ok!')

## 2. O algoritmo de Warshall

O subproblema restringe os vértices **intermediários**: $R^{(k)}[i][j] = 1$ se há caminho de $i$
até $j$ que, por dentro, só passa por vértices de $\{0,\dots,k-1\}$. Assim $R^{(0)}$ é a matriz de
adjacência e $R^{(n)}$ é o fecho. Liberar o vértice $k$ cria um caminho novo de $i$ até $j$
exatamente quando $i$ já chegava em $k$ e $k$ já chegava em $j$:

$$R^{(k+1)}[i][j] = R^{(k)}[i][j] \;\lor\; \bigl(R^{(k)}[i][k] \land R^{(k)}[k][j]\bigr).$$

O programa abaixo lê o grafo, roda Warshall em **uma única matriz** e imprime a matriz depois de
cada rodada, com as linhas rotuladas `R<k>.<i>:`.

**Fonte:** Warshall (1962); Cormen et al. (2009), §25.2; Levitin (2012), §8.4.

In [ ]:
%%writefile src/warshall.c
/* Algoritmo de WARSHALL: fecho transitivo de um digrafo.
   Uso: ./warshall < entrada     (n m e depois m linhas "u v", vertices 0..n-1)
   Imprime R(0), R(1), ..., R(n): a linha i de R(k) sai como "R<k>.<i>: ...".
   Warshall (1962); Cormen et al. (2009), sec. 25.2; Levitin (2012), sec. 8.4. */
#include <stdio.h>
#include <stdlib.h>

#define MAX 500

static char r[MAX][MAX];

static void imprimeMatriz( int k, int n) {
   for (int i = 0; i < n; ++i) {
      printf( "R%d.%d:", k, i);
      for (int j = 0; j < n; ++j) printf( " %d", r[i][j]);
      printf( "\n");
   }
}

void warshall( int n, int mostra) {
   if (mostra) imprimeMatriz( 0, n);
   for (int k = 0; k < n; ++k) {          /* libera o vertice k */
      for (int i = 0; i < n; ++i)
         if (r[i][k])                     /* i chega em k ... */
            for (int j = 0; j < n; ++j)
               if (r[k][j])               /* ... e k chega em j */
                  r[i][j] = 1;
      if (mostra) imprimeMatriz( k+1, n);
   }
}

int main( void) {
   int n, m, u, v;
   if (scanf( "%d %d", &n, &m) != 2 || n > MAX) return 1;
   for (int a = 0; a < m; ++a) {
      if (scanf( "%d %d", &u, &v) != 2) return 1;
      r[u][v] = 1;
   }
   warshall( n, n <= 12);
   int pares = 0;
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) pares += r[i][j];
   printf( "n=%d pares_alcancaveis=%d\n", n, pares);
   return 0;
}

In [ ]:
# Exemplo: Warshall no grafo das aulas de grafos (o mesmo dos slides).
saida = rodar('warshall.c', entrada=entrada_grafo(N_F, ARCOS_F), mostrar=False)
Rk = [matriz(saida, f'R{k}.', N_F) for k in range(N_F + 1)]
for k in (0, 2, 4, 6):
    print(f'R({k}):', ' | '.join(''.join(map(str, lin)) for lin in Rk[k]))
assert Rk[N_F] == R_F, 'O fecho de Warshall deve coincidir com o das buscas'
assert Rk[3] == Rk[2] and Rk[5] == Rk[4], 'Liberar 2 e 4 não muda nada'
print(saida.splitlines()[-1])

In [ ]:
# Visualização: as matrizes R(k) rodada a rodada; em laranja, os pares que acabaram de aparecer.
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import ListedColormap

fig, eixos = plt.subplots(1, N_F + 1, figsize=(15, 2.6))
for k, ax in enumerate(eixos):
    M = np.array(Rk[k])
    if k > 0:
        M = M + (M - np.array(Rk[k - 1]))      # 0 = sem caminho, 1 = antigo, 2 = novo
    ax.imshow(M, cmap=ListedColormap(['white', 'tab:blue', 'tab:orange']), vmin=0, vmax=2)
    ax.set_title('$R^{(0)}=A$' if k == 0 else f'libera {k - 1}', fontsize=10)
    ax.set_xticks(range(N_F)); ax.set_yticks(range(N_F))
    ax.tick_params(labelsize=7)
plt.suptitle('Warshall no grafo 0-1 0-5 1-0 1-5 2-4 3-1 5-3', y=1.05)
plt.show()

### Exercício 2: Warshall com a recorrência

Complete `warshall()` no programa abaixo. Ao contrário da versão dos slides, ela recebe a matriz
como parâmetro (`r[][MAX]`). Use **uma única matriz** e o laço de `k` por fora. O `main` roda o
grafo das aulas de grafos e depois compara sua resposta com uma DFS a partir de cada vértice em
300 digrafos aleatórios.

In [ ]:
%%writefile src/warshall_ex2.c
/* Exercicio 2: algoritmo de WARSHALL.
   r[i][j] comeca com a matriz de adjacencia e deve terminar com o fecho transitivo:
      R(k+1)[i][j] = R(k)[i][j] || (R(k)[i][k] && R(k)[k][j]).
   TODO: complete warshall(). O main confere com uma DFS a partir de cada vertice. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define MAX 30

void warshall( int n, char r[][MAX]) {
   /* TODO: tres lacos aninhados, com k (o vertice liberado) por fora */
}

/* referencia: DFS a partir de s, marcando em f[s][] os vertices alcancados */
static void dfs( int n, char a[][MAX], char f[][MAX], int s, int v) {
   for (int w = 0; w < n; ++w)
      if (a[v][w] && !f[s][w]) { f[s][w] = 1; dfs( n, a, f, s, w); }
}

int main( void) {
   static char a[MAX][MAX], r[MAX][MAX], f[MAX][MAX];
   int arcos[][2] = {{0,1},{0,5},{1,0},{1,5},{2,4},{3,1},{5,3}};
   memset( r, 0, sizeof r);
   for (int t = 0; t < 7; ++t) r[arcos[t][0]][arcos[t][1]] = 1;
   warshall( 6, r);
   for (int i = 0; i < 6; ++i) {
      printf( "F%d:", i);
      for (int j = 0; j < 6; ++j) printf( " %d", r[i][j]);
      printf( "\n");
   }
   srand( 2026);
   for (int teste = 0; teste < 300; ++teste) {
      int n = 1 + rand() % MAX, dens = 1 + rand() % 30;     /* densidade em % */
      memset( a, 0, sizeof a);  memset( f, 0, sizeof f);
      for (int i = 0; i < n; ++i)
         for (int j = 0; j < n; ++j)
            if (rand() % 100 < dens) a[i][j] = 1;
      memcpy( r, a, sizeof a);
      warshall( n, r);
      for (int s = 0; s < n; ++s) dfs( n, a, f, s, s);
      for (int i = 0; i < n; ++i)
         for (int j = 0; j < n; ++j)
            if (r[i][j] != f[i][j]) {
               printf( "ERRO: teste %d, n=%d: r[%d][%d]=%d, esperado %d\nok=0\n",
                       teste, n, i, j, r[i][j], f[i][j]);
               return 0;
            }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 2
saida = rodar('warshall_ex2.c')
assert matriz(saida, 'F', N_F) == R_F, 'No grafo das aulas de grafos, o fecho é o da Seção 1'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 2 ok!')

## 3. O algoritmo de Floyd

Com pesos nos arcos, $d^{(k)}_{ij}$ é o menor peso de um caminho de $i$ até $j$ com
intermediários em $\{0,\dots,k-1\}$, e a recorrência troca $\lor$ por $\min$ e $\land$ por $+$:

$$d^{(k+1)}_{ij} = \min\bigl(d^{(k)}_{ij},\ d^{(k)}_{ik} + d^{(k)}_{kj}\bigr),\qquad
d^{(0)}_{ii} = 0,\quad d^{(0)}_{ij} = w(i,j) \text{ ou } \infty.$$

O programa abaixo, além das distâncias, mantém a matriz **prox**: `prox[i][j]` é o primeiro vértice
depois de `i` no melhor caminho conhecido até `j`. Quando o vértice `k` melhora o par `(i,j)`, o
caminho passa a sair de `i` como saía para `k`, então `prox[i][j] = prox[i][k]`. Ao final, ele
olha a diagonal: se algum $d_{ii} < 0$, há **ciclo negativo**, e não imprime os caminhos.

O $\infty$ é `INT_MAX`; antes de somar, testamos se as duas parcelas são finitas. Sem esse teste,
`INT_MAX + w` estoura, e com pesos negativos "$\infty - 5$" pareceria um caminho.

**Fonte:** Floyd (1962); Cormen et al. (2009), §25.2 (Floyd-Warshall, matriz de predecessores e
Exercício 25.2-6); Levitin (2012), §8.4.

In [ ]:
%%writefile src/floyd.c
/* Algoritmo de FLOYD: caminhos minimos entre todos os pares, com reconstrucao.
   Uso: ./floyd < entrada     (n m e depois m linhas "u v w", vertices 0..n-1)
   Imprime D(0), ..., D(n) (linha i de D(k) como "D<k>.<i>: ..."), a matriz prox, os
   caminhos "cam_i_j=..." e "ciclo_negativo=0/1".
   Floyd (1962); Cormen et al. (2009), sec. 25.2; Levitin (2012), sec. 8.4. */
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>

#define MAX 500
#define INF INT_MAX     /* sem arco / sem caminho */

static int d[MAX][MAX], prox[MAX][MAX];

static void imprimeMatriz( const char *rot, int k, int n, int M[][MAX]) {
   for (int i = 0; i < n; ++i) {
      printf( "%s%d.%d:", rot, k, i);
      for (int j = 0; j < n; ++j)
         if (M[i][j] == INF) printf( " INF"); else printf( " %d", M[i][j]);
      printf( "\n");
   }
}

void floyd( int n, int mostra) {
   if (mostra) imprimeMatriz( "D", 0, n, d);
   for (int k = 0; k < n; ++k) {                 /* libera o vertice k */
      for (int i = 0; i < n; ++i) {
         if (d[i][k] == INF) continue;            /* i nao chega em k */
         for (int j = 0; j < n; ++j)
            if (d[k][j] != INF && d[i][k] + d[k][j] < d[i][j]) {
               d[i][j] = d[i][k] + d[k][j];
               prox[i][j] = prox[i][k];           /* sai de i como saia para k */
            }
      }
      if (mostra) imprimeMatriz( "D", k+1, n, d);
   }
}

int main( void) {
   int n, m, u, v, w;
   if (scanf( "%d %d", &n, &m) != 2 || n > MAX) return 1;
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) {
         d[i][j] = (i == j) ? 0 : INF;
         prox[i][j] = (i == j) ? i : -1;
      }
   for (int a = 0; a < m; ++a) {
      if (scanf( "%d %d %d", &u, &v, &w) != 3) return 1;
      if (w < d[u][v]) { d[u][v] = w;  prox[u][v] = v; }
   }
   floyd( n, n <= 12);
   int negativo = 0;
   for (int i = 0; i < n; ++i) if (d[i][i] < 0) negativo = 1;
   printf( "ciclo_negativo=%d\n", negativo);
   if (negativo || n > 12) return 0;
   imprimeMatriz( "prox", n, n, prox);
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) {
         if (i == j || prox[i][j] == -1) continue;
         printf( "cam_%d_%d=%d", i, j, i);
         for (int x = i; x != j; ) { x = prox[x][j];  printf( ",%d", x); }
         printf( "\n");
      }
   return 0;
}

In [ ]:
# Exemplo: o grafo do Levitin (a, b, c, d = 0, 1, 2, 3), o mesmo dos slides.
saida = rodar('floyd.c', entrada=entrada_grafo(N_L, ARCOS_L), mostrar=False)
Dk = [matriz(saida, f'D{k}.', N_L) for k in range(N_L + 1)]
nome = 'abcd'
for k, D in enumerate(Dk):
    print('D(0)' if k == 0 else f'libera {nome[k - 1]}', '|',
          '  '.join(' '.join('∞' if x == INF else str(x) for x in lin) for lin in D))
assert Dk[4] == [[0, 10, 3, 4], [2, 0, 5, 6], [7, 7, 0, 1], [6, 16, 9, 0]]
assert [D[2][0] for D in Dk] == [INF, INF, 9, 9, 7], 'd_ca: ∞ → 9 (via b) → 7 (via d)'
assert campo(saida, 'cam_1_3') == '1,0,2,3', 'b → a → c → d'
print('\ncaminho de b até d:', ' -> '.join(nome[int(x)] for x in campo(saida, 'cam_1_3').split(',')))

In [ ]:
# Exemplo: pesos negativos (Cormen, Figura 25.4) e um ciclo negativo.
saida = rodar('floyd.c', entrada=entrada_grafo(N_C, ARCOS_C), mostrar=False)
D_C = matriz(saida, f'D{N_C}.', N_C)
assert D_C == [[0, 1, -3, 2, -4], [3, 0, -4, 1, -1], [7, 4, 0, 5, 3],
               [2, -1, -5, 0, -2], [8, 5, 1, 6, 0]], 'Distâncias da Figura 25.4 do Cormen'
assert campo(saida, 'ciclo_negativo') == 0
print('Cormen, Figura 25.4 (vértices renumerados para 0..4):')
for lin in D_C:
    print(' '.join(f'{x:3d}' for x in lin))
print('caminho de 0 até 2:', campo(saida, 'cam_0_2'), '(peso -3, passando pelos arcos -4 e -5)')

# triângulo 0 -> 1 -> 2 -> 0 com peso 2 - 4 + 1 = -1
s = rodar('floyd.c', entrada=entrada_grafo(3, [(0, 1, 2), (1, 2, -4), (2, 0, 1)]), mostrar=False)
assert campo(s, 'ciclo_negativo') == 1
print('\ntriângulo de peso -1: diagonal final =', [matriz(s, 'D3.', 3)[i][i] for i in range(3)])

In [ ]:
%%writefile src/floyd_tempo.c
/* Mede o tempo do algoritmo de Floyd num grafo aleatorio denso com n vertices.
   Uso: ./floyd_tempo n */
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>
#include <time.h>

#define INF INT_MAX

int main( int argc, char *argv[]) {
   int n = atoi( argv[1]);
   int (*d)[n] = malloc( sizeof (int[n][n]));
   srand( n);
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j)
         d[i][j] = (i == j) ? 0 : (rand() % 2 ? 1 + rand() % 100 : INF);
   clock_t t0 = clock();
   for (int k = 0; k < n; ++k)
      for (int i = 0; i < n; ++i) {
         if (d[i][k] == INF) continue;
         for (int j = 0; j < n; ++j)
            if (d[k][j] != INF && d[i][k] + d[k][j] < d[i][j])
               d[i][j] = d[i][k] + d[k][j];
      }
   double t = (double) (clock() - t0) / CLOCKS_PER_SEC;
   long long soma = 0;
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) soma += d[i][j];
   printf( "n=%d t=%.4f soma=%lld\n", n, t, soma);
   free( d);
   return 0;
}

In [ ]:
# Visualização: o tempo de Floyd cresce como n^3 (escala log-log: reta de inclinação 3).
ns = [100, 150, 200, 300, 400, 600, 800]
ts = [campo(rodar('floyd_tempo.c', args=(n,), mostrar=False), 't') for n in ns]
incl = np.polyfit(np.log(ns[2:]), np.log(ts[2:]), 1)[0]

plt.figure(figsize=(6, 4))
plt.loglog(ns, ts, 'o-', label='Floyd (medido)')
plt.loglog(ns, [ts[-1] * (n / ns[-1]) ** 3 for n in ns], '--', label=r'$\propto n^3$')
plt.xlabel('n (vértices)')
plt.ylabel('tempo (s)')
plt.title(f'Floyd em grafos densos: inclinação ≈ {incl:.2f}')
plt.legend()
plt.grid(True, which='both', alpha=0.3)
plt.show()

### Exercício 3: reconstruindo o caminho

O `floyd()` abaixo já preenche `d` e `prox`. Complete `caminho()`: ela deve gravar em `v[]` os
vértices de um caminho mínimo de `i` até `j` (começando em `i` e terminando em `j`) e devolver
quantos vértices gravou, ou `-1` se `j` não é alcançável a partir de `i`. Para `i == j`, o caminho
é só `i`. O `main` confere, em 300 grafos aleatórios com pesos não negativos e para **todos** os
pares, que o caminho só usa arcos que existem e que a soma dos pesos é `d[i][j]`.

In [ ]:
%%writefile src/floyd_ex3.c
/* Exercicio 3: RECONSTRUCAO de caminhos minimos a partir da matriz prox.
   prox[i][j] = primeiro vertice depois de i num caminho minimo ate j (-1 se nao ha caminho).
   TODO: complete caminho(). O main confere todos os pares em grafos aleatorios. */
#include <stdio.h>
#include <stdlib.h>
#include <limits.h>

#define MAX 20
#define INF INT_MAX

static int w[MAX][MAX], d[MAX][MAX], prox[MAX][MAX];

void floyd( int n) {
   for (int i = 0; i < n; ++i)
      for (int j = 0; j < n; ++j) {
         d[i][j] = (i == j) ? 0 : w[i][j];
         prox[i][j] = (i == j) ? i : (w[i][j] == INF ? -1 : j);
      }
   for (int k = 0; k < n; ++k)
      for (int i = 0; i < n; ++i) {
         if (d[i][k] == INF) continue;
         for (int j = 0; j < n; ++j)
            if (d[k][j] != INF && d[i][k] + d[k][j] < d[i][j]) {
               d[i][j] = d[i][k] + d[k][j];
               prox[i][j] = prox[i][k];
            }
      }
}

/* grava em v[] um caminho minimo de i ate j; devolve o numero de vertices, ou -1 */
int caminho( int i, int j, int v[]) {
   /* TODO: siga prox[.][j] a partir de i ate chegar em j */
   return -1;
}

int main( void) {
   int v[MAX+1];
   /* Levitin: a b c d = 0 1 2 3 */
   for (int i = 0; i < 4; ++i) for (int j = 0; j < 4; ++j) w[i][j] = INF;
   w[0][2] = 3;  w[1][0] = 2;  w[2][1] = 7;  w[2][3] = 1;  w[3][0] = 6;
   floyd( 4);
   int t = caminho( 1, 3, v);
   printf( "levitin_b_d=");
   for (int x = 0; x < t; ++x) printf( x ? ",%d" : "%d", v[x]);
   printf( "\nlevitin_d_b_tam=%d\n", caminho( 3, 1, v));
   srand( 2026);
   for (int teste = 0; teste < 300; ++teste) {
      int n = 1 + rand() % MAX, dens = 5 + rand() % 40;
      for (int i = 0; i < n; ++i)
         for (int j = 0; j < n; ++j)
            w[i][j] = (i != j && rand() % 100 < dens) ? rand() % 20 : INF;
      floyd( n);
      for (int i = 0; i < n; ++i)
         for (int j = 0; j < n; ++j) {
            for (int x = 0; x <= MAX; ++x) v[x] = -7;
            int tam = caminho( i, j, v), ok = 1, soma = 0;
            if (d[i][j] == INF) ok = (tam == -1);
            else if (tam < 1 || tam > n || v[0] != i || v[tam-1] != j) ok = 0;
            else
               for (int x = 0; x + 1 < tam && ok; ++x)
                  if (w[v[x]][v[x+1]] == INF) ok = 0;
                  else soma += w[v[x]][v[x+1]];
            if (ok && d[i][j] != INF && soma != d[i][j]) ok = 0;
            if (!ok) {
               printf( "ERRO: teste %d, n=%d, par (%d,%d): d=%d, tam=%d\nok=0\n",
                       teste, n, i, j, d[i][j], tam);
               return 0;
            }
         }
   }
   printf( "ok=1\n");
   return 0;
}

In [ ]:
# Teste automático do Exercício 3
saida = rodar('floyd_ex3.c')
assert campo(saida, 'levitin_b_d') == '1,0,2,3', 'No Levitin, o caminho de b até d é b a c d'
assert campo(saida, 'levitin_d_b_tam') == 4, 'd → a → c → b tem 4 vértices'
assert campo(saida, 'ok') == 1, 'Verifique sua implementação (veja o ERRO impresso acima)'
print('Exercício 3 ok!')

## 4. A ordem dos laços

Nos slides, o laço de $k$ é o **externo**: a rodada $k$ termina para todos os pares $(i,j)$
antes de o vértice $k+1$ ser liberado. Um erro comum é escrever os laços na ordem $i, j, k$,
com $k$ por dentro. Essa versão calcula cada $d_{ij}$ uma única vez, como
$\min_k (d_{ik} + d_{kj})$, usando pedaços que podem ainda não estar prontos.

As duas versões estão em Python abaixo. A errada às vezes acerta. Por exemplo, no grafo do
Levitin ela dá a resposta certa.

**Fonte:** Cormen et al. (2009), §25.2; Dasgupta et al. (2008), §6.6.

In [ ]:
# Exemplo: Floyd certo (k por fora) e a versão com a ordem errada (k por dentro).
def matriz_pesos(n, arcos):
    W = [[0 if i == j else INF for j in range(n)] for i in range(n)]
    for u, v, w in arcos:
        W[u][v] = min(W[u][v], w)
    return W

def floyd_py(W):
    n, D = len(W), [lin[:] for lin in W]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                D[i][j] = min(D[i][j], D[i][k] + D[k][j])
    return D

def floyd_ordem_errada(W):
    n, D = len(W), [lin[:] for lin in W]
    for i in range(n):
        for j in range(n):
            for k in range(n):
                D[i][j] = min(D[i][j], D[i][k] + D[k][j])
    return D

W_L = matriz_pesos(N_L, ARCOS_L)
print('Levitin, ordem certa  :', floyd_py(W_L))
print('Levitin, ordem errada :', floyd_ordem_errada(W_L))
print('Coincidem?', floyd_py(W_L) == floyd_ordem_errada(W_L))

### Exercício 4: um contraexemplo para a ordem errada

Complete `contraexemplo()`, que devolve a matriz de pesos `W` (com `INF` onde não há arco e `0`
na diagonal) de um digrafo **com no máximo 5 vértices e pesos positivos** em que
`floyd_ordem_errada(W)` erra alguma distância.

Dica: a versão errada termina a linha $i$ inteira antes de olhar a linha $i+1$. Pense num
caminho $0 \to \dots \to j$ cujo segundo pedaço $k \leadsto j$ só ficaria pronto numa linha
**posterior** a $0$. Escreva também, num comentário, qual par sai errado e por quê.

In [ ]:
# Exercício 4: contraexemplo para a ordem dos laços
# TODO: devolva uma matriz de pesos (até 5 vértices, pesos positivos) em que a ordem errada falha.

def contraexemplo():
    # TODO: implemente aqui
    pass


# Teste automático
W = contraexemplo()
assert W is not None, 'Implemente contraexemplo()'
n = len(W)
assert 2 <= n <= 5 and all(len(lin) == n for lin in W), 'Matriz quadrada com até 5 vértices'
assert all(W[i][i] == 0 for i in range(n)), 'A diagonal deve ser 0'
assert all(W[i][j] == INF or W[i][j] > 0 for i in range(n) for j in range(n) if i != j), 'Pesos positivos'
certo, errado = floyd_py(W), floyd_ordem_errada(W)
assert certo != errado, 'Neste grafo a ordem errada ainda acerta: procure outro'
for i in range(n):
    for j in range(n):
        if certo[i][j] != errado[i][j]:
            print(f'par ({i},{j}): ordem certa = {certo[i][j]}, ordem errada = {errado[i][j]}')
print('Exercício 4 ok!')

In [ ]:
# Visualização: com que frequência a ordem errada erra, em grafos aleatórios de n vértices?
random.seed(2026)
ns_err, frac = list(range(3, 13)), []
for n in ns_err:
    erros = 0
    for _ in range(300):
        arcos = [(u, v, random.randint(1, 9)) for u in range(n) for v in range(n)
                 if u != v and random.random() < 0.3]
        W = matriz_pesos(n, arcos)
        erros += floyd_py(W) != floyd_ordem_errada(W)
    frac.append(erros / 300)

plt.figure(figsize=(6, 3.5))
plt.plot(ns_err, frac, 'o-')
plt.xlabel('n (vértices, densidade 30%)')
plt.ylabel('fração de grafos com erro')
plt.title('Ordem i, j, k: erra cada vez mais à medida que n cresce')
plt.ylim(0, 1)
plt.grid(True, alpha=0.3)
plt.show()

## Desafio Final: o caminho de maior gargalo

Numa rede de computadores, cada enlace $(u,v)$ tem uma **capacidade** $c(u,v) > 0$ (em Mb/s). A
capacidade de um caminho é a do seu enlace **mais estreito** (o gargalo). Para transmitir um
arquivo grande de $i$ para $j$, queremos o caminho de **maior gargalo**:

$$b(i,j) = \max_{\text{caminhos } p:\, i \leadsto j}\ \ \min_{(u,v) \in p}\ c(u,v).$$

Nos slides vimos que Warshall e Floyd são o mesmo laço triplo com operações diferentes. Aqui a
"soma" de dois pedaços é o $\min$ e a escolha entre caminhos é o $\max$.

1. **Recorrência.** Defina $b^{(k)}_{ij}$ como no Floyd, com intermediários em
   $\{0,\dots,k-1\}$. Escreva a recorrência e os valores iniciais. Quanto vale $b^{(0)}_{ij}$
   quando não há arco? E $b_{ii}$?
2. **Subestrutura ótima.** Por que aqui não é preciso nenhuma hipótese como "sem ciclos
   negativos"? (Pense no que um ciclo faz com o gargalo de um caminho.)
3. **Implementação.** Escreva `src/gargalo.c`, que lê `n m` e os arcos `u v c` e imprime a
   matriz $b$ e, para um par dado, o caminho (use uma matriz *prox*, como no Exercício 3).
4. **Verificação.** Para grafos aleatórios com até 8 vértices, compare com uma força bruta em
   Python que enumera todos os **caminhos simples** de $i$ até $j$.
5. **Ligação com Warshall.** Se todas as capacidades forem 1, o que a sua matriz $b$ calcula?
6. **Para ir além.** O caminho de maior gargalo entre quaisquer dois vértices de um grafo
   **não dirigido** sempre está contido numa árvore geradora **máxima** (Kruskal com as
   arestas em ordem decrescente). Confira essa propriedade nos seus testes. Ela volta na
   Unidade 3.

In [ ]:
# Desafio: caminho de maior gargalo
# TODO: resolva o desafio abaixo
# Sugestão: copie src/floyd.c para src/gargalo.c; "sem arco" passa a ser 0 (capacidade nula),
# a diagonal deixa de ser 0 e a comparação d[i][k] + d[k][j] < d[i][j] vira um max de mins.

# Rede de exemplo (u, v, capacidade em Mb/s):
REDE = [(0, 1, 10), (0, 2, 4), (1, 2, 3), (1, 3, 2), (2, 3, 8), (3, 4, 6), (2, 4, 1), (4, 0, 5)]

# Sua solução aqui

## Referências

Veja o arquivo `../referencias.bib` para a lista completa. As principais para este tutorial:

- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009. §25.2 (Floyd-Warshall e fecho transitivo), §25.1, §25.3. (Em português: *Algoritmos: Teoria e Prática*, Elsevier, 2012.)
- LEVITIN, A. *Introduction to the Design and Analysis of Algorithms*. 3. ed. Pearson, 2012. §8.4.
- DASGUPTA, S.; PAPADIMITRIOU, C. H.; VAZIRANI, U. *Algorithms*. McGraw-Hill, 2008. §6.6.
- WARSHALL, S. A theorem on Boolean matrices. *Journal of the ACM*, 9(1):11–12, 1962.
- FLOYD, R. W. Algorithm 97: Shortest path. *Communications of the ACM*, 5(6):345, 1962.